In [ ]:
# ============ ONE CELL: install + build + simulate + render ============
!pip -q install -U mujoco mediapy scipy
!rm -rf mujoco_menagerie
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/google-deepmind/mujoco_menagerie.git
!cd mujoco_menagerie && git sparse-checkout set unitree_g1

import os
os.environ["MUJOCO_GL"] = "egl"          # use "osmesa" if no GPU (slower)
import numpy as np, mujoco, mediapy as media
from scipy.interpolate import CubicSpline
from IPython.display import Video, display

# ---------------- parameters ----------------
T, D, N, T0 = 0.60, 0.30, 10, 1.0     # flight time, hold time, no. of tosses, start delay
FPS, W, H   = 60, 1280, 720
R_APPLE, OFF = 0.038, 0.045
X_FWD, Y_SIDE, Z_REL = 0.24, 0.17, 0.12   # palm rest pose relative to pelvis (closer => bent elbow)
ELBOW_START = 1.2                          # initial elbow bend (rad) so IK picks the bent solution
PALM_N = {0: np.array([0, -1., 0]), 1: np.array([0, 1., 0])}
g = 9.81

# ---------------- model (G1 + apple + palm sites) ----------------
spec = mujoco.MjSpec.from_file("mujoco_menagerie/unitree_g1/scene.xml")
for s in ("left", "right"):
    spec.body(f"{s}_wrist_yaw_link").add_site(name=f"palm_{s}", pos=[0.08, 0, 0], size=[0.01, 0, 0])
apple = spec.worldbody.add_body(name="apple", pos=[0, 0, 3])
apple.add_freejoint()
apple.add_geom(type=mujoco.mjtGeom.mjGEOM_SPHERE, size=[R_APPLE, 0, 0], mass=0.2,
               rgba=[0.80, 0.05, 0.05, 1], contype=0, conaffinity=0)
apple.add_geom(type=mujoco.mjtGeom.mjGEOM_CAPSULE, size=[0.004, 0.012, 0], pos=[0, 0, R_APPLE + 0.006],
               mass=0.001, rgba=[0.3, 0.2, 0.05, 1], contype=0, conaffinity=0)
m = spec.compile(); d = mujoco.MjData(m)
m.vis.global_.offwidth, m.vis.global_.offheight = W, H
m.vis.quality.offsamples = 8
mujoco.mj_resetData(m, d); mujoco.mj_forward(m, d)
base = d.qpos[:3].copy()

names = ["shoulder_pitch", "shoulder_roll", "shoulder_yaw", "elbow", "wrist_roll", "wrist_pitch", "wrist_yaw"]
SIDES = ["left", "right"]
QA = [[m.joint(f"{s}_{n}_joint").qposadr[0] for n in names] for s in SIDES]
DA = [[m.joint(f"{s}_{n}_joint").dofadr[0] for n in names] for s in SIDES]
LO = [np.array([m.joint(f"{s}_{n}_joint").range[0] for n in names]) for s in SIDES]
HI = [np.array([m.joint(f"{s}_{n}_joint").range[1] for n in names]) for s in SIDES]
SID = [m.site(f"palm_{s}").id for s in SIDES]
try: WAIST = m.joint("waist_yaw_joint").qposadr[0]
except Exception: WAIST = None
AQ = m.jnt_qposadr[m.body("apple").jntadr[0]]

# ---------------- motion plan (hand offsets: dx forward, dy sideways, dz up) ----------------
C = T + D
tosses = [(T0 + i * C, i % 2) for i in range(N)]       # (time, tossing hand) 0=L 1=R
T_END = tosses[-1][0] + T + 1.5
SGN = {0: 1.0, 1: -1.0}
ev = {0: [], 1: []}
for t, h in tosses:
    c, tc = 1 - h, t + T
    # thrower: pull in + dip (elbow flexes) -> extend + flick up at release -> follow-through
    ev[h] += [(t - 0.12, -0.06, 0.03, -0.11), (t, 0.04, 0.0, 0.05), (t + 0.15, 0.0, -0.03, 0.10)]
    # catcher: reach out + up (elbow opens) -> catch -> pull in + down to cushion (elbow flexes)
    ev[c] += [(tc - 0.12, 0.05, -0.02, 0.08), (tc, 0.02, 0.0, 0.03), (tc + 0.12, -0.06, 0.03, -0.11)]

def make_spline(h):
    pts = sorted([(0, 0, 0, 0)] + ev[h] + [(T_END, 0, 0, 0)])
    ts, ys = [], []
    for t, dx, dy, dz in pts:
        if ts and t - ts[-1] < 0.04: continue
        ts.append(t); ys.append([dx, dy, dz])
    return CubicSpline(ts, np.array(ys), bc_type="clamped")
SPL = [make_spline(0), make_spline(1)]

def hand_pos(h, t):
    dx, dy, dz = SPL[h](min(t, T_END))
    return base + np.array([X_FWD + dx, SGN[h] * (Y_SIDE + dy), Z_REL + dz])

flights = []
for t, h in tosses:
    flights.append((hand_pos(h, t) + [0, 0, OFF], hand_pos(1 - h, t + T) + [0, 0, OFF]))

# ---------------- IK ----------------
UP = np.array([0, 0, 1.0]); WR = 0.4
jp, jr = np.zeros((3, m.nv)), np.zeros((3, m.nv))
q_rest = [None, None]
def ik(h, target, iters=15, damp=0.05):
    qa, da, sid = QA[h], DA[h], SID[h]
    for _ in range(iters):
        mujoco.mj_kinematics(m, d); mujoco.mj_comPos(m, d)
        mujoco.mj_jacSite(m, d, jp, jr, sid)
        a = d.site_xmat[sid].reshape(3, 3) @ PALM_N[h]
        e = np.concatenate([target - d.site_xpos[sid], WR * np.cross(a, UP)])
        J = np.vstack([jp[:, da], WR * jr[:, da]])
        Jp = J.T @ np.linalg.inv(J @ J.T + damp**2 * np.eye(6))
        rest = q_rest[h] if q_rest[h] is not None else 0 * d.qpos[qa]
        dq = Jp @ e + (np.eye(7) - Jp @ J) @ (0.08 * (rest - d.qpos[qa]))
        d.qpos[qa] = np.clip(d.qpos[qa] + dq, LO[h], HI[h])

for h in (0, 1):                       # settle into a bent-elbow ready pose
    d.qpos[QA[h][3]] = ELBOW_START
    ik(h, hand_pos(h, 0), iters=300); q_rest[h] = d.qpos[QA[h]].copy()

# ---------------- apple ----------------
def apple_state(t):
    for i, (ti, h) in enumerate(tosses):
        if ti <= t < ti + T:
            tau = t - ti; A0, A1 = flights[i]
            v = (A1 - A0) / T + np.array([0, 0, 0.5 * g * T])
            p = A0 + v * tau - np.array([0, 0, 0.5 * g * tau**2])
            q = np.zeros(4); mujoco.mju_axisAngle2Quat(q, np.array([0, 1., 0]), 2 * np.pi * tau / T)
            return p, q
    holder = next((h for ti, h in tosses if ti > t), 1 - tosses[-1][1])
    n = d.site_xmat[SID[holder]].reshape(3, 3) @ PALM_N[holder]
    return d.site_xpos[SID[holder]] + n * OFF, np.array([1., 0, 0, 0])

# ---------------- render ----------------
cam = mujoco.MjvCamera()
cam.lookat[:] = base + [0.15, 0, 0.22]; cam.distance = 1.45; cam.azimuth = 155; cam.elevation = -8
out = "g1_apple_toss.mp4"
renderer = mujoco.Renderer(m, H, W)
elbow_log = [[], []]
with media.VideoWriter(out, shape=(H, W), fps=FPS, bps=12_000_000) as vw:
    for k in range(int(T_END * FPS)):
        t = k / FPS
        if WAIST is not None: d.qpos[WAIST] = 0.07 * np.sin(np.pi * (t - T0) / C)
        for h in (0, 1):
            ik(h, hand_pos(h, t)); elbow_log[h].append(d.qpos[QA[h][3]])
        mujoco.mj_forward(m, d)
        p, q = apple_state(t)
        d.qpos[AQ:AQ + 3] = p; d.qpos[AQ + 3:AQ + 7] = q
        mujoco.mj_forward(m, d)
        renderer.update_scene(d, cam); vw.add_image(renderer.render())
renderer.close()
for h, s in enumerate(SIDES):
    print(f"{s} elbow range: {np.degrees(min(elbow_log[h])):.0f}° to {np.degrees(max(elbow_log[h])):.0f}°")
display(Video(out, embed=True, width=900, html_attributes="controls loop autoplay"))

Output hidden; open in https://colab.research.google.com to view.